## 0. 제출 정보
- 이름: 박재선
- GitHub ID: lilypark0930-lab
- 작성일: 2026.09.20
- 최종 제출 URL: https://github.com/lilypark0930-lab/llm-data-analysis-study/blob/main/chapter03/chapter03.ipynb

## 1. 데이터 로딩과 구조 확인
### 실행/결과
- 4개 CSV 로딩 여부: True
- 각 데이터 shape: 
customers (150, 6)
products (100, 4)
orders (300, 5)
order_items (764, 5)
- 주요 컬럼: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
- dtypes에서 주목한 컬럼: signup_date 

### Evidence
![데이터 구조 확인](images/step01_structure.png)

### 결과 관찰
직접 확인한 사실을 작성하세요.
- 데이터의 행과 열 개수 : 각 데이터별 행과 열의 개수를 shape을 통해 알 수 있다. 
- 실제 컬럼명 : 주요 컬럽을 알수 있다. 
- 눈에 띄는 값의 형태를 미리 알 수 있다.

### 나의 해석과 판단
분석 전에 특히 주의해야 할 데이터셋/컬럼과 이유를 작성하세요.
- signup_date 와 같이 날짜처럼 보이지만 문자열(str)로 읽힌 컬럼이 있다.

### 업무·분석적 의미
구조를 먼저 확인하지 않고 분석을 시작할 때 생길 수 있는 문제를 작성하세요.
- 데이터의 오류나 결측치를 확인하지 않고 다음 프로세스를 진행하여 지속적인 오류를 발생시킬 수 있다.

### 한계와 추가 확인 사항
아직 알 수 없는 품질 문제를 작성하세요.
- 결측, 중복 등의 세부 데이터 패턴 오류 문제

## 2. 결측·중복·키 품질
- 주요 ID 결측: 없음
- 주요 ID 중복: 없음
- 전체 행 중복: 없음

![결측 중복 점검](images/step02_quality.png)

### 결과 관찰
결측치와 중복은 없음

### 나의 해석과 판단
어떤 문제를 먼저 처리해야 하는지 우선순위를 작성하세요.
order_items.order_id는 한 주문에 여러 상품이 포함될 수 있기 때문에 반복될 수 있기에, 중복이 나와도 지우면 안됨.

### 업무·분석적 의미
기준 키에 결측과 중복이 없어서 고객·주문·상품 단위 집계에서 같은 대상이 두 번 세어질 위험이 낮다. 다음 단계인 테이블 결합과 관계 검증도 안정적으로 진행할 수 있다.
- 만약 orders.order_id가 중복되었다면 주문 건수와 매출이 부풀려지고, 결합 시 행이 불어난다. 결측이 있었다면 결합에서 빠지는 행이 생겨 분석 대상이 조용히 줄어든다.
- 중복이 없다는 결과는 "정상 데이터"라는 뜻이 아니라 "기계적으로 겹치는 행이 없다"는 뜻이다. 같은 주문이 다른 ID로 이중 입력된 경우처럼 의미상의 중복은 이 점검으로 잡히지 않는다.

### 한계와 추가 확인 사항
각 데이터의 특성과 연관 관계를 이해하고 있어야 중복과 결측치에 대한 올바른 판단으로 후행 작업을 올바르게 진행할 수 있음

## 3. 숫자형·범주형·날짜 점검
- 숫자형 범위에서 주목한 값: order_items.quantity는 1~5(평균 약 3.05, 중앙값 3), unit_price는 5,000~200,000원(평균 약 108,562원, 중앙값 111,000원)  
- 범주형 빈도에서 주목한 값: order_status는 completed 184건(61.3%), cancelled 64건(21.3%), refunded 52건(17.3%). 취소·환불 합계가 116건(약 38.7%)
- 날짜 변환 실패 건수: 0
- 날짜 범위: 가장 빠른 날짜: 2025-09-15 00:00:00 가장 최근 날짜: 2026-09-14 00:00:00

![기본 분포와 날짜 확인](images/step03_distribution.png)

### 결과 관찰
- order_items 764행 모두 quantity, unit_price에 값이 있고(count 764), quantity는 1~5, unit_price는 5,000~200,000 범위 안에 있다. 
- unit_price의 평균(약 10.9만)과 중앙값(11.1만)이 비슷해 한쪽으로 크게 치우치지 않았고, 최소·최대값도 극단적으로 튀지 않는다.
- 주문 300건 중 취소와 환불이 약 39%를 차지한다. 주문 상태 값은 3종류뿐이며 표기 불일치나 결측은 없다.
- order_date는 전부 날짜로 변환되었고(실패 0건), 미래 날짜나 비정상적으로 오래된 날짜 없이 1년 범위 안에 있다.

### 나의 해석과 판단
이상해 보이는 값이 실제 오류인지 업무적으로 가능한 값인지 구분하기 위해 무엇을 더 확인해야 하는지 작성하세요.
- 취소·환불 비중(약 39%): 데이터가 원래 그런 것인지, 상태 코드의 정의가 다른 것인지 업무 기준을 확인한다. 이 비중이 크면 매출·주문 분석에서 어떤 상태를 포함할지에 따라 결과가 크게 달라진다.
- unit_price 5,000원과 200,000원: products.price와 비교해 실제 상품 가격 범위 안의 값인지, 주문 시점 가격과 현재 가격이 다른 것인지 확인한다.
- 날짜: 월별 주문 건수를 보고 특정 달에 데이터가 비어 있거나 몰려 있지 않은지 확인한다.

### 업무·분석적 의미
- 주문 상태는 orders에만 있고 order_items에는 없다. 그래서 상품·금액 기준 매출을 계산하려면 orders와 결합한 뒤 어떤 상태를 매출로 인정할지 먼저 정해야 한다. completed만 쓸지, 환불을 차감할지에 따라 매출이 달라진다.
- quantity × unit_price로 라인 금액을 만들 수 있으며, 주문 단위 집계는 order_id 기준으로 묶어서 계산한다.
- 분석 기간이 정확히 1년치라 월별 추이는 볼 수 있지만, 계절성은 한 주기만 관찰할 수 있어 전년 대비 비교는 어렵다.

### 한계와 추가 확인 사항
- unit_price와 products.price의 일치 여부, 취소·환불 주문의 상세 사유는 이번 점검에서 확인하지 못했다.
- customers.signup_date는 아직 문자열이므로 날짜 변환 후 가입일이 주문일보다 늦은 고객이 있는지 점검이 필요하다.

## 4. CSV 간 키 관계 검증
- 없는 `customer_id`: 0건
- 없는 `order_id`: 0건
- 없는 `product_id`: 0건

![PK FK 관계 검증](images/step04_relationship.png)

### 결과 관찰
- 세 가지 키 관계(orders.customer_id → customers, order_items.order_id → orders, order_items.product_id → products)에서 부모 테이블에 없는 값은 모두 0건이다.
- 고아 행이 없어서 고객 → 주문 → 상품 라인 → 상품으로 이어지는 연결이 끊기지 않는다.

### 나의 해석과 판단
키 관계 문제가 발견되었다면 바로 삭제하면 안 되는 이유를 작성하세요.
- 원인이 입력 오류인지 마스터 파일의 추출 시점 차이인지 알 수 없고, 삭제하면 실제 매출과 주문 이력이 사라지며, orders를 지우면 그 주문의 order_items까지 고아 행이 되어 정합성이 다시 깨지기 때문이다. 먼저 해당 행을 확인하고 원인을 파악한 뒤 보존, 플래그 표시, 분석 제외 중에서 선택하고 건수와 기준을 기록해야 한다.

### 업무·분석적 의미
- 키 관계가 온전하므로 orders–order_items–products–customers를 결합해 고객/상품/기간별 분석을 진행할 수 있다.
- 결합해도 매칭되지 않는 행이 없어서, 결합 과정에서 분석 대상이 조용히 빠질 위험은 낮다.
- 관계가 1:N이므로 결합 시 주문 단위 값이 상품 행마다 복제된다. 결합 전후 행 수를 비교하고, 주문 단위 값은 집계한 뒤에 합산해야 한다.

### 한계와 추가 확인 사항
- 이번 검증은 키 값이 존재하는지만 확인한 것이며, 연결된 값이 업무적으로 맞는지는 확인하지 못했다. 예를 들어 unit_price와 products.price의 일치 여부, 가입일 이전 주문 여부는 별도로 점검해야 한다.
- 반대 방향(주문 없는 고객, 판매 이력 없는 상품, 상품 라인 없는 주문)은 확인하지 않았다.


## 5. LLM 구조 설명 검증
- LLM에 제공한 Safe Context: 온라인 쇼핑몰 데이터는 customers, products, orders, order_items의 4개 테이블로 구성되어 있다. customers는 150행이며 customer_id가 고유하고, age·city·signup_date를 포함한다. 각 테이블의 주요 ID는 결측이나 중복이 없으며, 테이블 간 연결에 사용하는 customer_id·order_id·product_id도 모두 참조 대상 테이블에 존재한다.
- LLM이 제안한 추가 점검: 데이터 타입 및 날짜 형식, 비주요 컬럼의 결측 패턴, 연령·수량·가격의 허용 범위, city·category·order_status의 표기 통일성, 가입일과 주문일의 시간 순서, 주문과 주문항목 간 완전성, 주문 상태별 취소·환불 처리 기준, 상품 기준가와 실제 판매단가의 차이, 주문 금액 계산의 정합성, 업무상 중복 주문, 이상치, 데이터 기간의 연속성,, 고객 프로필 분포, 그리고 매출·주문·고객 수의 분석 기준 정의를 제안하였다.
- 실제 데이터에서 확인한 항목: customers의 customer_id, products의 product_id, orders의 order_id, order_items의 order_item_id는 모두 고유했다. 주요 ID 컬럼의 결측값과 중복값은 없었다. 또한 orders의 customer_id, order_items의 order_id 및 product_id는 모두 연결 대상 테이블에 존재하여 기본적인 참조 무결성이 확보되었다
- 채택/수정/보류한 내용: 우선 채택할 항목은 주문 상태별 처리 기준, 주문-주문항목 완전성, 날짜 논리성, 수량·단가 이상치, 범주형 값 표준화이다. 고객 프로필 분포와 업무상 중복 주문 탐지는 분석 목적과 주문 데이터의 시간 단위가 정해진 뒤 보류한다.

![LLM 구조 검토](images/step05_llm.png)

### 나의 해석과 판단
LLM 제안 중 가장 유용했던 것과 가장 조심해야 할 것을 작성하세요.
LLM 제안 중 가장 유용했던 것은 주문 상태별 처리 기준을 먼저 정하라는 제안이다. 취소/환불/결제 대기 주문을 매출과 구매 고객 수에 포함하는 방식에 따라 이후의 핵심 분석 결과가 달라질 수 있기 때문이다.
### 한계와 추가 확인 사항
현재 점검은 주요 ID의 결측/중복과 테이블 간 참조 무결성에 집중되어 있다. 따라서 비주요 컬럼의 결측, 값의 범위, 날짜 형식, 상태값 표기, 수량·단가 이상치 등은 실제 데이터 기준으로 추가 확인이 필요하다. 또한 할인액, 배송비, 세금, 환불 금액, 가격 이력 관련 컬럼의 존재 여부도 확인해야 순매출과 실제 판매가격을 정확하게 해석할 수 있다.
## 6. Chapter 03 최종 판단
### 데이터의 첫인상 3가지
1.네 개의 테이블이 고객, 상품, 주문, 주문항목의 구조로 잘 분리되어 있어 고객·상품·매출 분석을 수행하기에 적절한 관계형 데이터 구조를 갖추고 있다.
2.주요 ID의 결측과 중복이 없고 테이블 간 연결 대상도 모두 존재하므로, 기본적인 데이터 무결성은 비교적 양호한 것으로 판단된다.
3.주문 상태·가격·날짜·수량에 대한 업무 규칙을 확인해야 신뢰할 수 있는 분석이 가능하다.

### 다음 Chapter 전에 반드시 확인/처리해야 할 항목
1.완료/취소/환불/결제 대기 등 주문 상태별 분석 포함 기준을 정의한다.
2.가입일과 주문일의 시간 순서, 날짜 형식, 수량·단가·연령의 비정상값을 점검한다.
3.city, category, order_status의 오타·공백·표기 불일치를 정리하고, 주문과 주문항목 간 누락 여부를 확인한다.

### 현재 데이터만으로 단정할 수 없는 것
매출이 실제로 증가하거나 감소했는지와 그 원인
특정 고객군이나 상품 카테고리가 성과가 좋거나 나쁜 이유
할인/취소/환불을 반영한 실제 순매출 및 수익성
고객의 장기 재구매 성향

## 최종 제출 체크
- [O] Notebook을 처음부터 끝까지 실행했습니다.
- [O] 오류 셀이 남아 있지 않습니다.
- [O] 핵심 Evidence를 첨부했습니다.
- [O] 관찰과 해석을 구분했습니다.
- [O] 개인정보/Secret이 없습니다.
- [O] `chapter03/chapter03.ipynb`가 GitHub에서 정상 표시됩니다.
- [O] 최종 Notebook 파일 URL을 제출합니다.